# DSAB review experiments: GPU runs

Runs the GPU half of the review experiments on the team's repo
(`boracandan/Algoverse_Truth_Directions_Research`). The CPU half (verdict parsing, item
matching, cosine summary, baseline table) was already run locally.

**Runtime:** Runtime → Change runtime type → **A100** (best) or **L4**. A T4 does not have
enough memory for an 8B model in fp16; the first cell stops if it sees one.

**How to run:** Runtime → Run all, or run cell by cell. Sections 3, 6 and 4 matter most;
5 and 7 can wait for a second session.

**If Colab disconnects:** reconnect and run the setup cells (1 to 4) again, then rerun the
experiment cell that was interrupted. Every result is saved to Drive as soon as it is
computed, and finished tasks are skipped automatically.

**Results:** `MyDrive/dsab_review_results/review_results.csv` plus figures and tables in the
same folder. Section 9 prints the summaries and downloads a zip.

Rough time on an A100: sanity check 5 min, exp3 ~45 min, exp6 ~1 h, exp4 ~1.5 h,
exp5 ~1 h, exp7 ~1.5 h.

## 1. Check the GPU

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> A100 or L4."
name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}: {mem_gb:.0f} GB")
assert mem_gb >= 22, f"{name} has {mem_gb:.0f} GB; an 8B model in fp16 needs an A100 or L4 (22 GB+)."
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

## 2. Drive folder for results

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

RESULTS = '/content/drive/MyDrive/dsab_review_results'   # survives disconnects
CACHE = '/content/dsab_cache'                              # local disk; activations are too big for Drive
os.makedirs(RESULTS, exist_ok=True)
os.environ['DSAB_RESULTS_DIR'] = RESULTS
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('results ->', RESULTS)

## 3. Get the team's repo

Pinned to commit `60c8954` (their last push, 9/13), the version the review was done on.
The 1.4 GB cosine file is skipped; it is not needed here.

In [ ]:
REPO = '/content/Algoverse_Truth_Directions_Research'
if not os.path.exists(REPO):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/boracandan/Algoverse_Truth_Directions_Research.git {REPO}
%cd {REPO}
!git checkout -q 60c8954 && git --no-pager log --oneline -1
!mkdir -p experiments/review_experiments
!pip install -q tabulate
import torch, transformers, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)

## 4. Write the experiment code into the repo

Each cell below writes one file into `experiments/review_experiments/`. Run them; no need
to read them. They are the same files as on the `archana-review-experiments` branch,
tested end to end before this notebook was made.

In [ ]:
%%writefile experiments/review_experiments/common.py
"""
Shared helpers for the review experiments (exp3-exp9).

Everything here is lifted from the existing notebooks so results stay comparable with
results_database.csv:
  - extract_all_layer_activations is the Figure_3_Cross_Condition_Generalization version
    (length-sorted batches, left padding, final-token residual stream at every layer).
  - fit_linear_probe is the same bias-free, mean-centred probe (Adam, lr 1e-3,
    weight_decay 0.1, 1000 full-batch steps), with one change: it takes a seed.

What is new:
  - a seed on every probe, plus a bootstrap confidence interval on every AUROC. The seed
    only changes the probe's random init, and full-batch Adam lands in nearly the same
    place every time, so seed spread on its own will look tiny. The bootstrap over test
    items is the error bar that reflects how few test items there are (~290 for A1-A3).
  - a results file with more key columns (experiment, readout, seed, eval target, subset)
    so reruns with different seeds or readouts sit side by side instead of overwriting
    each other the way drop_duplicates(keep='last') does in results_database.csv.

GPU-only imports (torch, transformers) are deferred into the functions that need them so
the CPU experiments (exp8, exp9, and the data-prep halves of exp3/exp6) run on a laptop.
"""

import gc
import os
import re
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

REPO_ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = REPO_ROOT / "datasets"
# DSAB_RESULTS_DIR lets a Colab run write straight to Drive, so a disconnect loses nothing.
RESULTS_DIR = Path(os.environ.get("DSAB_RESULTS_DIR", Path(__file__).resolve().parent / "results"))
RESULTS_CSV = RESULTS_DIR / "review_results.csv"

TASKS = ["A1", "A2", "A3", "F0", "F1", "F2", "F3", "F4", "F5"]
ARITH_TASKS = ["A1", "A2", "A3"]
FACT_TASKS = ["F0", "F1", "F2", "F3", "F4", "F5"]

DEEPSEEK_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
DEEPSEEK_SHORT = "deepseek-r1-distill-8b"  # same label results_database.csv uses
DEEPSEEK_PAD_ID = 128001  # pad id hardcoded in the Figure 3 notebook

# Same folders as CONDITION_DICT in the existing notebooks.
CONDITION_DICT = {
    "no-prompt": DATA_DIR / "plain_dataset",
    "cot-zero-shot": DATA_DIR / "CoT_datasets" / "lexically_cleaned",
    "sentence-based-CoT": DATA_DIR / "CoT_datasets" / "sentence_based_lexically_cleaned",
    "no-prompt-chat-template": DATA_DIR / "plain_template_dataset",
    "ablation-filler-token": DATA_DIR / "ablation_datasets" / "filler_token_only",
    "ablation-instructions-and-template": DATA_DIR / "ablation_datasets" / "instructions_and_template",
    "ablation-instructions-only": DATA_DIR / "ablation_datasets" / "instructions_only",
}

DEEPSEEK_MARKERS = ["<｜begin▁of▁sentence｜>", "<｜User｜>"]


# ---------------------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------------------

def to_binary_labels(series):
    """Labels are stored as True/False in most files and 0/1 in others (F5). pandas reads
    True/False as bool already; strings are handled too in case a file was re-saved."""
    mapping = {"true": 1, "false": 0, "1": 1, "0": 0}
    if series.dtype == bool:
        return series.astype(int).to_numpy()
    return series.map(lambda v: mapping[str(v).strip().lower()]).astype(int).to_numpy()


def load_split(condition, task, split):
    path = CONDITION_DICT[condition] / f"{task}_{split}.csv"
    df = pd.read_csv(path)
    df["y"] = to_binary_labels(df["label"])
    return df


def statement_text(row):
    """The bare statement for any condition's row. Plain files have a `statement` column;
    every other file starts its text with the statement (after DeepSeek's BOS/User markers
    when a chat template was used) and follows it with a blank line or <｜Assistant｜>."""
    if "statement" in row and isinstance(row["statement"], str):
        return row["statement"].strip()
    text = row.get("generated_statement_texts", row.get("extracted_statement_texts"))
    for marker in DEEPSEEK_MARKERS:
        text = text.replace(marker, "")
    return re.split(r"\n\n|<｜Assistant｜>", text, maxsplit=1)[0].strip()


def ids_for(df, tokenizer=None):
    """Token-id lists for each row. Plain files carry no ids, so they are tokenized here
    exactly as the Figure 3 notebook does (tokenizer default, which adds BOS)."""
    if "extracted_statement_ids" in df.columns:
        return [literal_eval(x) if isinstance(x, str) else list(x) for x in df["extracted_statement_ids"]]
    if tokenizer is None:
        raise ValueError("plain dataset rows need a tokenizer to produce ids")
    return tokenizer(df["statement"].tolist())["input_ids"]


# ---------------------------------------------------------------------------------------
# Model + activations (GPU)
# ---------------------------------------------------------------------------------------

def load_model(model_id=DEEPSEEK_ID):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    kwargs = dict(device_map="cuda" if torch.cuda.is_available() else "cpu", trust_remote_code=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, **kwargs)
    except TypeError:  # transformers < 4.56 calls it torch_dtype
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, **kwargs)
    model.eval()
    return model, tokenizer


def extract_all_layer_activations(id_lists, model, pad_token_id, batch_size=4, desc="Extracting"):
    """Final-token residual stream at every layer, one array per layer, rows in input order.

    Same algorithm as the Figure 3 notebook: sort by length, left-pad, read [:, -1, :],
    un-sort. Readout position is controlled by the caller truncating each id list, which
    is exact for a causal model: the activation at position k only sees tokens 0..k, so
    reading the last token of ids[:k+1] equals reading position k of the full sequence.
    """
    import torch
    from tqdm.auto import tqdm

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tensors = [torch.tensor(ids, dtype=torch.long) for ids in id_lists]
    lengths = torch.tensor([len(t) for t in tensors])
    sort_idx = torch.argsort(lengths)
    inverse_idx = torch.argsort(sort_idx)
    sorted_tensors = [tensors[i] for i in sort_idx.tolist()]

    n_layers = model.config.num_hidden_layers + 1
    per_layer = [[] for _ in range(n_layers)]
    device = next(model.parameters()).device

    with tqdm(total=len(sorted_tensors), desc=desc, leave=False) as pbar:
        for i in range(0, len(sorted_tensors), batch_size):
            batch = sorted_tensors[i:i + batch_size]
            padded = torch.nn.utils.rnn.pad_sequence(
                batch, batch_first=True, padding_value=pad_token_id, padding_side="left"
            ).to(device)
            # Mask by length, not by `padded != pad_token_id`: the pad id can also be a real
            # token inside a sequence (Granite reuses EOS as pad).
            lens = torch.tensor([len(t) for t in batch], device=device)
            positions = torch.arange(padded.shape[1], device=device).unsqueeze(0)
            attention_mask = (positions >= (padded.shape[1] - lens).unsqueeze(1)).long()
            with torch.no_grad():
                out = model(input_ids=padded, attention_mask=attention_mask, output_hidden_states=True)
            for layer_idx, hidden in enumerate(out.hidden_states):
                per_layer[layer_idx].append(hidden[:, -1, :].float().cpu())
            del out, padded, attention_mask
            pbar.update(len(batch))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return [torch.cat(acts, dim=0)[inverse_idx].numpy() for acts in per_layer]


def cached_activations(cache_dir, key, compute_fn):
    """Optional disk cache (float16 .npz) keyed by a string the caller builds from model,
    condition, task, split and readout. Several experiments read the same activations
    (plain and sentence-based-CoT at the final token), so pointing them all at one cache
    folder on Drive saves most of the GPU time after the first run."""
    if cache_dir is None:
        return compute_fn()
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f"{key}.npz"
    if path.exists():
        data = np.load(path)
        return [data[f"l{i}"].astype(np.float32) for i in range(len(data.files))]
    acts = compute_fn()
    np.savez(path, **{f"l{i}": a.astype(np.float16) for i, a in enumerate(acts)})
    return acts


# ---------------------------------------------------------------------------------------
# Probes + scoring
# ---------------------------------------------------------------------------------------

def fit_linear_probe(X_train, y_train, seed=0):
    """Bias-free logistic probe on mean-centred activations (as in the notebooks), seeded."""
    import torch
    import torch.nn as nn

    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X = torch.tensor(X_train, dtype=torch.float32)
    train_mean = X.mean(dim=0)
    Xc = (X - train_mean).to(device)
    y = torch.tensor(y_train, dtype=torch.float32, device=device)

    probe = nn.Linear(X.shape[1], 1, bias=False).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(1000):
        opt.zero_grad()
        loss_fn(probe(Xc).squeeze(-1), y).backward()
        opt.step()
    return probe.weight.detach().cpu().numpy().flatten(), train_mean.numpy()


def probe_scores(w, train_mean, X):
    return (X - train_mean) @ w


def _auroc_rows(y, scores):
    """AUROC for every row of 2-D (n_boot, n) arrays at once, via the Mann-Whitney rank
    formula with average ranks for ties (the same value roc_auc_score gives)."""
    ranks = rankdata(scores, axis=1)
    n_pos = y.sum(axis=1)
    n_neg = y.shape[1] - n_pos
    with np.errstate(invalid="ignore", divide="ignore"):
        return ((ranks * y).sum(axis=1) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def auroc_with_ci(y, scores, n_boot=1000, seed=0):
    """AUROC plus a 95% percentile bootstrap interval over test items. Returns NaNs when
    the subset has a single class (AUROC undefined) instead of the 0.5 the notebooks use,
    so an undefined number can never be read as chance. The bootstrap is vectorized: a
    loop of 1000 roc_auc_score calls per row made exp3 alone take about an hour of CPU."""
    y = np.asarray(y).astype(float)
    scores = np.asarray(scores, dtype=float)
    if len(np.unique(y)) < 2:
        return float("nan"), float("nan"), float("nan")
    point = roc_auc_score(y, scores)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), (n_boot, len(y)))
    boots = _auroc_rows(y[idx], scores[idx])
    boots = boots[np.isfinite(boots)]  # resamples that drew a single class
    if not len(boots):
        return float(point), float("nan"), float("nan")
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(point), float(lo), float(hi)


def parse_layers(arg, n_layers):
    """--layers all | 8,12,16 | 0-32:4"""
    if arg in (None, "all"):
        return list(range(n_layers))
    if ":" in arg:
        span, step = arg.split(":")
        lo, hi = span.split("-")
        return list(range(int(lo), min(int(hi), n_layers - 1) + 1, int(step)))
    return [int(x) for x in arg.split(",") if int(x) < n_layers]


# ---------------------------------------------------------------------------------------
# Results file
# ---------------------------------------------------------------------------------------

RESULT_KEYS = [
    "experiment", "model", "train_task", "test_task", "train_condition", "test_condition",
    "readout", "layer", "seed", "eval_target", "subset",
]
RESULT_COLUMNS = RESULT_KEYS + ["n_train", "n_test", "auroc", "ci_low", "ci_high"]


def append_results(rows, csv_path=None):
    """Append rows, replacing only rows that match on every key column (so a rerun of the
    same seed and readout replaces itself, and a new seed adds rows)."""
    if not rows:
        return
    csv_path = Path(csv_path or RESULTS_CSV)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame(rows)[RESULT_COLUMNS]
    if csv_path.exists():
        old = pd.read_csv(csv_path)
        new = pd.concat([old, new], ignore_index=True).drop_duplicates(subset=RESULT_KEYS, keep="last")
    new.to_csv(csv_path, index=False)


def already_done(experiment, **match):
    """True when the results file already has rows for this experiment and every
    column=value in `match`. Lets a rerun after a Colab disconnect skip finished work."""
    if not Path(RESULTS_CSV).exists():
        return False
    df = pd.read_csv(RESULTS_CSV)
    mask = df["experiment"] == experiment
    for col, val in match.items():
        mask &= df[col].astype(str) == str(val)
    return bool(mask.any())


def result_row(**kw):
    row = {c: kw.get(c, "") for c in RESULT_COLUMNS}
    row.update({k: v for k, v in kw.items() if k in RESULT_COLUMNS})
    return row

In [ ]:
%%writefile experiments/review_experiments/exp3_probe_vs_verdict.py
"""
Experiment 3: does the probe read the truth, or the verdict the model is about to give?

The sentence-based CoT prefixes stop right before the model's first verdict word. By then
the model has usually decided, so a probe there could be reading "truth" or "the answer I
am about to write". On most items the two agree (the model is right ~90-99% of the time),
so ordinary AUROC cannot tell them apart. They only come apart on the items where the
model's final answer is WRONG.

On those disagreement items, AUROC against the true label is the test:
  > 0.5  the probe sides with the truth, against the model's own answer
  < 0.5  the probe sides with the model's (wrong) answer
On that subset the verdict is exactly 1 - label, so AUROC vs verdict = 1 - AUROC vs label
and only the label number is reported.

Step 1 (CPU, no model): parse the model's answer out of every generated trace and write
datasets/CoT_datasets/verdicts/{task}.csv. Run with --parse-only on a laptop.

Step 2 (GPU): 5-fold cross-validation over train+test pooled, so every item gets an
out-of-fold score and all disagreement items are usable (the test split alone has ~50 in
F0, and almost none in A1-A3). Grouped by statement so a duplicated statement never sits
on both sides of a fold. Three probes per fold and layer:
  cot_label    trained on CoT activations with the true label      (the paper's probe)
  cot_verdict  trained on CoT activations with the model's answer  (a verdict probe)
  plain_label  trained on plain-statement activations, true label  (the transfer probe),
               scored both on plain activations and on CoT activations of held-out items

Arithmetic has almost no model errors, so this experiment can only settle the factual
tasks. For arithmetic, use exp4 (earlier readout points) instead.

Usage:
  python experiments/review_experiments/exp3_probe_vs_verdict.py --parse-only
  python experiments/review_experiments/exp3_probe_vs_verdict.py --tasks F0 F3 F4 F5 \
      --layers all --cache-dir /content/drive/MyDrive/dsab_cache
"""

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402

COT = "sentence-based-CoT"
PLAIN = "no-prompt"
VERDICT_DIR = C.DATA_DIR / "CoT_datasets" / "verdicts"


# ---------------------------------------------------------------------------------------
# Step 1: parse verdicts
# ---------------------------------------------------------------------------------------

def parse_verdict(generated_text):
    """Returns (final_answer, think_verdict) as 1/0/None.

    The prompt itself contains the template lines 'Answer: Yes' / 'Answer: No' and
    'The statement is true/false', so only text after <｜Assistant｜> is searched. The
    final 'Answer:' line is the model's committed answer. The think verdict only counts
    the exact line the prompt asks for ('The statement is true.' on its own line): a
    looser 'statement is true' match picks up hypotheticals mid-reasoning ('if X, the
    statement is true') and produced dozens of false conflicts on F3/F4.
    """
    gen = generated_text.split("<｜Assistant｜>", 1)[-1]
    answers = re.findall(r"Answer:\s*\**\s*(Yes|No)\b", gen, flags=re.IGNORECASE)
    thinks = re.findall(r"^\s*\**the statement is\s+(true|false)\.?\**\s*$", gen,
                        flags=re.IGNORECASE | re.MULTILINE)
    final = None if not answers else int(answers[-1].lower() == "yes")
    think = None if not thinks else int(thinks[-1].lower() == "true")
    return final, think


def build_verdicts(tasks):
    VERDICT_DIR.mkdir(parents=True, exist_ok=True)
    summary = []
    for task in tasks:
        parts = []
        for split in ["train", "test"]:
            df = C.load_split(COT, task, split)
            parsed = df["generated_statement_texts"].map(parse_verdict)
            out = pd.DataFrame({
                "split": split,
                "row": np.arange(len(df)),
                "statement": [C.statement_text(r) for _, r in df.iterrows()],
                "label": df["y"],
                "answer_verdict": [p[0] for p in parsed],
                "think_verdict": [p[1] for p in parsed],
            })
            # Committed answer first; fall back to the in-thinking line if the answer
            # line is missing (e.g. generation cut off before </think>).
            out["model_verdict"] = out["answer_verdict"].fillna(out["think_verdict"])
            out["verdict_source"] = np.where(out["answer_verdict"].notna(), "answer_line",
                                             np.where(out["think_verdict"].notna(), "think_line", "none"))
            parts.append(out)
        out = pd.concat(parts, ignore_index=True)
        out.to_csv(VERDICT_DIR / f"{task}.csv", index=False)

        has = out["model_verdict"].notna()
        both = out["answer_verdict"].notna() & out["think_verdict"].notna()
        summary.append({
            "task": task,
            "items": len(out),
            "parsed": int(has.sum()),
            "model_accuracy": round(float((out.loc[has, "model_verdict"] == out.loc[has, "label"]).mean()), 3),
            "disagree_total": int((has & (out["model_verdict"] != out["label"])).sum()),
            "disagree_test_split": int((has & (out["model_verdict"] != out["label"]) & (out["split"] == "test")).sum()),
            "think_vs_answer_conflicts": int((both & (out["answer_verdict"] != out["think_verdict"])).sum()),
        })
    summary = pd.DataFrame(summary)
    print(summary.to_string(index=False))
    C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(C.RESULTS_DIR / "exp3_verdict_summary.csv", index=False)
    return summary


# ---------------------------------------------------------------------------------------
# Step 2: probes
# ---------------------------------------------------------------------------------------

def plain_rows_for(statements, task, tokenizer):
    """Plain-statement ids for the same statements, in the same order (exp6 matching)."""
    # Read the full plain files directly: this is a lookup table, never subsampled.
    plain = pd.concat([pd.read_csv(C.CONDITION_DICT[PLAIN] / f"{task}_{s}.csv") for s in ["train", "test"]],
                      ignore_index=True)
    lookup = dict(zip(plain["statement"].str.strip(), plain["statement"]))
    missing = [s for s in statements if s not in lookup]
    if missing:
        raise ValueError(f"{task}: {len(missing)} CoT statements have no plain match, e.g. {missing[0]!r}")
    return tokenizer([lookup[s] for s in statements])["input_ids"]


def run_probes(tasks, layers_arg, seeds, folds, cache_dir, batch_size):
    model, tokenizer = C.load_model(C.DEEPSEEK_ID)
    n_layers = model.config.num_hidden_layers + 1
    layers = C.parse_layers(layers_arg, n_layers)

    for task in tasks:
        todo = [s for s in seeds if not C.already_done("exp3_probe_vs_verdict", train_task=task, seed=s)]
        if not todo:
            print(f"\n{task}: already done, skipping (delete its rows from the results file to redo)")
            summarize(task)
            continue
        v = pd.read_csv(VERDICT_DIR / f"{task}.csv")
        cot = pd.concat([C.load_split(COT, task, s) for s in ["train", "test"]], ignore_index=True)
        assert len(cot) == len(v), "verdict file out of sync with CoT data; rerun --parse-only"

        keep = v["model_verdict"].notna().to_numpy()
        v, cot = v[keep].reset_index(drop=True), cot[keep].reset_index(drop=True)
        y = v["label"].to_numpy().astype(int)
        verdict = v["model_verdict"].to_numpy().astype(int)
        disagree = verdict != y
        print(f"\n{task}: {len(v)} items, {disagree.sum()} where the model's answer is wrong")

        cot_ids = C.ids_for(cot)
        plain_ids = plain_rows_for(v["statement"].tolist(), task, tokenizer)
        acts_cot = C.cached_activations(
            cache_dir, f"{C.DEEPSEEK_SHORT}__{COT}__{task}__all__final",
            lambda: C.extract_all_layer_activations(cot_ids, model, C.DEEPSEEK_PAD_ID, batch_size, f"{task} CoT"))
        acts_plain = C.cached_activations(
            cache_dir, f"{C.DEEPSEEK_SHORT}__{PLAIN}-matched__{task}__all__final",
            lambda: C.extract_all_layer_activations(plain_ids, model, C.DEEPSEEK_PAD_ID, batch_size, f"{task} plain"))

        groups = pd.factorize(v["statement"])[0]
        rows = []
        for seed in todo:
            # Stratify on label x verdict so each fold gets its share of the rare
            # disagreement items.
            strat = y * 2 + verdict
            splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
            fold_of = np.empty(len(v), dtype=int)
            for k, (_, te) in enumerate(splitter.split(np.zeros(len(v)), strat, groups)):
                fold_of[te] = k

            for layer in layers:
                Xc, Xp = acts_cot[layer], acts_plain[layer]
                oof = {name: np.zeros(len(v)) for name in
                       ["cot_label", "cot_verdict", "plain_label_on_plain", "plain_label_on_cot"]}
                for k in range(folds):
                    tr, te = fold_of != k, fold_of == k
                    w, m = C.fit_linear_probe(Xc[tr], y[tr], seed)
                    oof["cot_label"][te] = C.probe_scores(w, m, Xc[te])
                    w, m = C.fit_linear_probe(Xc[tr], verdict[tr], seed)
                    oof["cot_verdict"][te] = C.probe_scores(w, m, Xc[te])
                    w, m = C.fit_linear_probe(Xp[tr], y[tr], seed)
                    oof["plain_label_on_plain"][te] = C.probe_scores(w, m, Xp[te])
                    oof["plain_label_on_cot"][te] = C.probe_scores(w, m, Xc[te])

                for name, scores in oof.items():
                    train_cond = PLAIN if name.startswith("plain") else COT
                    test_cond = PLAIN if name.endswith("on_plain") else COT
                    for subset, mask in [("all", np.ones(len(v), bool)),
                                         ("model_wrong", disagree),
                                         ("model_right", ~disagree)]:
                        for target, labels in [("label", y), ("verdict", verdict)]:
                            if subset == "model_wrong" and target == "verdict":
                                continue  # = 1 - (vs label) on this subset
                            auc, lo, hi = C.auroc_with_ci(labels[mask], scores[mask], seed=seed)
                            rows.append(C.result_row(
                                experiment="exp3_probe_vs_verdict", model=C.DEEPSEEK_SHORT,
                                train_task=task, test_task=task,
                                train_condition=f"{train_cond}[{'verdict' if name == 'cot_verdict' else 'label'}]",
                                test_condition=test_cond, readout="final", layer=layer, seed=seed,
                                eval_target=target, subset=subset,
                                n_train=int(round(len(v) * (folds - 1) / folds)), n_test=int(mask.sum()),
                                auroc=auc, ci_low=lo, ci_high=hi))
            C.append_results(rows)
            rows = []
        summarize(task)


def summarize(task):
    """Per probe: choose the layer on ALL items (best AUROC vs the true label), then report
    the model-wrong AUROC at that layer. Choosing the layer on the model-wrong items
    themselves (e.g. the layer furthest from 0.5) would overstate the result."""
    df = pd.read_csv(C.RESULTS_CSV)
    df = df[(df["experiment"] == "exp3_probe_vs_verdict") & (df["train_task"] == task) & (df["eval_target"] == "label")]
    if df.empty:
        return
    print(f"\n{task}: AUROC vs TRUE label on items the model got wrong "
          f"(>0.5 = follows truth, <0.5 = follows the model's answer); layer chosen on all items")
    for (tr, te), g in df.groupby(["train_condition", "test_condition"]):
        g = g.groupby(["subset", "layer"])[["auroc", "ci_low", "ci_high", "n_test"]].mean()
        layer = g.loc["all", "auroc"].idxmax()
        a, w = g.loc[("all", layer)], g.loc[("model_wrong", layer)]
        print(f"  {tr:>30} -> {te:<20} layer {layer:>2}: all items {a.auroc:.3f} | "
              f"model wrong {w.auroc:.3f} [{w.ci_low:.3f}, {w.ci_high:.3f}] n={int(w.n_test)}")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.TASKS)
    ap.add_argument("--parse-only", action="store_true", help="only build the verdict files (CPU)")
    ap.add_argument("--layers", default="all", help="all | 8,16,24 | 0-32:4")
    ap.add_argument("--seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--cache-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    args = ap.parse_args()

    build_verdicts(args.tasks)
    if not args.parse_only:
        run_probes(args.tasks, args.layers, args.seeds, args.folds, args.cache_dir, args.batch_size)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments/review_experiments/exp4_early_readout.py
"""
Experiment 4: when, during the reasoning, does truth become readable?

The sentence-based CoT prefixes are read at one point only: the token before the verdict.
For arithmetic that is after the model has already written the answer in words
("so (13 * 6) - 37 equals 41. Wait a minute, the right side is 37"), which is why A2/A3
reach AUROC 1.000. This reads the same traces at fractions of the way through the
reasoning, so the paper can show the curve instead of just the end point.

Fractions are of the reasoning tokens (prompt_len .. end of extracted prefix):
  0.00  end of the prompt (<think>\\n). Same input as ablation-instructions-and-template.
  0.25, 0.50, 0.75
  1.00  the current pre-verdict readout. Should reproduce results_database.csv.

For each fraction:
  - in-domain probe (train/test at that fraction), with bootstrap CI
  - plain -> fraction transfer (plain-statement probe scored at that fraction): does the
    no-CoT truth direction start to fire partway through, or only at the end?
  - TF-IDF bag-of-words on the decoded prefix text at that fraction (CPU, same settings
    as lexical_shortcut_baseline.py). Where the probe climbs but TF-IDF does not, the
    probe is reading more than the words on the page.

Usage:
  python experiments/review_experiments/exp4_early_readout.py --tasks A1 A2 A3 F3 F4 F5 \
      --fractions 0 0.25 0.5 0.75 1 --layers 0-32:2 --cache-dir /content/drive/MyDrive/dsab_cache
  python experiments/review_experiments/exp4_early_readout.py --tfidf-only   # CPU only
"""

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402

COT = "sentence-based-CoT"
PLAIN = "no-prompt"


def truncate(df, frac):
    """Id lists cut at `frac` of the way through the reasoning part of each prefix."""
    ids = C.ids_for(df)
    out = []
    for row_ids, prompt_len in zip(ids, df["prompt_len"].astype(int)):
        n_reason = len(row_ids) - prompt_len
        out.append(row_ids[: prompt_len + int(round(frac * n_reason))])
    return out


def tfidf_auroc(train_texts, y_tr, test_texts, y_te, seed):
    """Mirrors evaluate_tfidf_bow in lexical_shortcut_baseline.py (unigrams, lowercase,
    LogisticRegression). Only the reasoning text is used: the prompt is identical across
    items apart from the statement, so including it adds the statement's own words."""
    vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 1), token_pattern=r"(?u)\b[A-Za-z0-9']+\b")
    Xtr = vec.fit_transform(train_texts)
    Xte = vec.transform(test_texts)
    if Xtr.shape[1] == 0:
        return float("nan"), float("nan"), float("nan")
    clf = LogisticRegression(max_iter=1000, random_state=seed).fit(Xtr, y_tr)
    return C.auroc_with_ci(y_te, clf.predict_proba(Xte)[:, 1], seed=seed)


def run_tfidf(tasks, fractions, seeds, tokenizer):
    rows = []
    for task in tasks:
        tr, te = C.load_split(COT, task, "train"), C.load_split(COT, task, "test")
        for frac in fractions:
            if C.already_done("exp4_early_readout", model="tfidf-bow", train_task=task, readout=f"frac={frac:.2f}"):
                continue
            texts = []
            for df in (tr, te):
                ids = truncate(df, frac)
                texts.append([tokenizer.decode(i[p:]) for i, p in zip(ids, df["prompt_len"].astype(int))])
            for seed in seeds:
                if frac == 0:
                    auc = lo = hi = float("nan")  # no reasoning text yet
                else:
                    auc, lo, hi = tfidf_auroc(texts[0], tr["y"], texts[1], te["y"], seed)
                rows.append(C.result_row(
                    experiment="exp4_early_readout", model="tfidf-bow", train_task=task, test_task=task,
                    train_condition=COT, test_condition=COT, readout=f"frac={frac:.2f}", layer=-1, seed=seed,
                    eval_target="label", subset="all", n_train=len(tr), n_test=len(te),
                    auroc=auc, ci_low=lo, ci_high=hi))
            print(f"  {task} frac={frac:.2f} tfidf AUROC {rows[-1]['auroc']:.3f}")
            C.append_results(rows)
            rows = []


def run_probes(tasks, fractions, layers_arg, seeds, cache_dir, batch_size, model, tokenizer):
    n_layers = model.config.num_hidden_layers + 1
    layers = C.parse_layers(layers_arg, n_layers)
    for task in tasks:
        tr, te = C.load_split(COT, task, "train"), C.load_split(COT, task, "test")
        ptr = C.load_split(PLAIN, task, "train")
        plain_tr = C.cached_activations(
            cache_dir, f"{C.DEEPSEEK_SHORT}__{PLAIN}__{task}__train__final",
            lambda: C.extract_all_layer_activations(C.ids_for(ptr, tokenizer), model, C.DEEPSEEK_PAD_ID,
                                                    batch_size, f"{task} plain train"))
        for frac in fractions:
            if C.already_done("exp4_early_readout", model=C.DEEPSEEK_SHORT, train_task=task,
                              readout=f"frac={frac:.2f}"):
                print(f"  {task} frac={frac:.2f}: already done, skipping")
                continue
            acts = {}
            for split, df in (("train", tr), ("test", te)):
                ids = truncate(df, frac)
                acts[split] = C.cached_activations(
                    cache_dir, f"{C.DEEPSEEK_SHORT}__{COT}__{task}__{split}__frac{frac:.2f}",
                    lambda ids=ids, split=split: C.extract_all_layer_activations(
                        ids, model, C.DEEPSEEK_PAD_ID, batch_size, f"{task} {split} frac={frac:.2f}"))
            rows = []
            for seed in seeds:
                for layer in layers:
                    w, m = C.fit_linear_probe(acts["train"][layer], tr["y"].to_numpy(), seed)
                    auc, lo, hi = C.auroc_with_ci(te["y"], C.probe_scores(w, m, acts["test"][layer]), seed=seed)
                    rows.append(C.result_row(
                        experiment="exp4_early_readout", model=C.DEEPSEEK_SHORT, train_task=task, test_task=task,
                        train_condition=COT, test_condition=COT, readout=f"frac={frac:.2f}", layer=layer,
                        seed=seed, eval_target="label", subset="all", n_train=len(tr), n_test=len(te),
                        auroc=auc, ci_low=lo, ci_high=hi))
                    w, m = C.fit_linear_probe(plain_tr[layer], ptr["y"].to_numpy(), seed)
                    auc, lo, hi = C.auroc_with_ci(te["y"], C.probe_scores(w, m, acts["test"][layer]), seed=seed)
                    rows.append(C.result_row(
                        experiment="exp4_early_readout", model=C.DEEPSEEK_SHORT, train_task=task, test_task=task,
                        train_condition=PLAIN, test_condition=COT, readout=f"frac={frac:.2f}", layer=layer,
                        seed=seed, eval_target="label", subset="all", n_train=len(ptr), n_test=len(te),
                        auroc=auc, ci_low=lo, ci_high=hi))
            C.append_results(rows)
            best = max((r for r in rows if r["train_condition"] == COT), key=lambda r: r["auroc"])
            print(f"  {task} frac={frac:.2f}: best in-domain {best['auroc']:.3f} at layer {best['layer']}")
    plot(tasks)


def plot(tasks):
    import matplotlib.pyplot as plt

    df = pd.read_csv(C.RESULTS_CSV)
    df = df[df["experiment"] == "exp4_early_readout"].copy()
    df["frac"] = df["readout"].str.replace("frac=", "").astype(float)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
    series = [
        ("In-domain probe (best layer)", (df["model"] == C.DEEPSEEK_SHORT) & (df["train_condition"] == COT)),
        ("Plain probe -> CoT (best layer)", (df["model"] == C.DEEPSEEK_SHORT) & (df["train_condition"] == PLAIN)),
        ("TF-IDF on reasoning text", df["model"] == "tfidf-bow"),
    ]
    for ax, (title, mask) in zip(axes, series):
        sub = df[mask]
        for task in tasks:
            t = sub[sub["train_task"] == task].groupby(["frac", "layer"])["auroc"].mean().groupby("frac").max()
            if len(t):
                ax.plot(t.index, t.values, marker="o", label=task)
        ax.axhline(0.5, color="black", linestyle=":", alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel("Fraction of reasoning read")
        ax.grid(True, linestyle="--", alpha=0.35)
    axes[0].set_ylabel("AUROC")
    axes[-1].legend(fontsize=8)
    out = C.RESULTS_DIR / "exp4_early_readout.png"
    plt.tight_layout()
    plt.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved {out}")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.TASKS)
    ap.add_argument("--fractions", nargs="+", type=float, default=[0.0, 0.25, 0.5, 0.75, 1.0])
    ap.add_argument("--layers", default="all")
    ap.add_argument("--seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--cache-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--tfidf-only", action="store_true", help="CPU only: skip the model")
    args = ap.parse_args()

    if args.tfidf_only:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(C.DEEPSEEK_ID)
        run_tfidf(args.tasks, args.fractions, args.seeds, tokenizer)
        plot(args.tasks)
        return
    model, tokenizer = C.load_model(C.DEEPSEEK_ID)
    run_tfidf(args.tasks, args.fractions, args.seeds, tokenizer)
    run_probes(args.tasks, args.fractions, args.layers, args.seeds, args.cache_dir, args.batch_size, model, tokenizer)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments/review_experiments/exp5_readout_position.py
"""
Experiment 5: separate the effect of the readout POSITION from the effect of the CONDITION.

Each condition is currently read at a different token:
  no-prompt                           last token of the statement
  no-prompt-chat-template             <think>\\n, after the template wrapped round the statement
  ablation-instructions-and-template  <think>\\n, after statement + instructions + template
  ablation-instructions-only          last token of the instructions
  ablation-filler-token               last '.' of the filler block
  sentence-based-CoT                  token before the verdict

So when A1 goes 0.933 (no-prompt) -> 0.619 (chat template) -> 0.51 (instructions), it is
not clear whether the instructions hurt or the readout simply moved off the statement.
This reads every condition at the same anchors:

  statement_end  last token of the statement (or the token that contains its final
                 character, when the tokenizer merges it with what follows, e.g. '.\\n\\n')
  prompt_end     end of the user turn (<think>\\n for templated conditions; the full
                 input for conditions without a template, where there is no separate turn)
  final          the current readout (should reproduce results_database.csv)

The model is causal, so the activation at statement_end only depends on tokens up to the
statement: every condition that shares the same prefix there gives the SAME activation.
That is a built-in check, and the summary prints it. The informative comparison is at
prompt_end, where conditions differ only in what sits between statement and readout.

Usage:
  python experiments/review_experiments/exp5_readout_position.py --tasks A1 A2 A3 F3 F4 F5 \
      --layers 0-32:2 --cache-dir /content/drive/MyDrive/dsab_cache
"""

import argparse
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402

CONDITIONS = [
    "no-prompt",
    "no-prompt-chat-template",
    "ablation-instructions-only",
    "ablation-instructions-and-template",
    "ablation-filler-token",
    "sentence-based-CoT",
]
THINK_ID = 128013  # <think>
NEWLINE_ID = 198   # \n


def statement_end_index(ids, statement, tokenizer):
    """Index of the token holding the statement's last character: the shortest prefix whose
    decoded text contains the whole statement. Binary search on prefix length."""
    lo, hi = 1, len(ids)
    if statement not in tokenizer.decode(ids):
        return None
    while lo < hi:
        mid = (lo + hi) // 2
        if statement in tokenizer.decode(ids[:mid]):
            hi = mid
        else:
            lo = mid + 1
    return lo - 1


def prompt_end_index(ids, prompt_len=None):
    """Last prompt token: '<think>\\n' when present, else prompt_len - 1 for CoT rows, else
    the last token (instructions-only and plain have no template, so the whole input is
    the prompt)."""
    for i in range(len(ids) - 1):
        if ids[i] == THINK_ID and ids[i + 1] == NEWLINE_ID:
            return i + 1
    if prompt_len is not None:
        return int(prompt_len) - 1
    return len(ids) - 1


def cut_ids(df, readout, tokenizer):
    ids = C.ids_for(df, tokenizer)
    out, misses = [], 0
    for k, row_ids in enumerate(ids):
        if readout == "final":
            out.append(row_ids)
            continue
        if readout == "statement_end":
            idx = statement_end_index(row_ids, C.statement_text(df.iloc[k]), tokenizer)
            if idx is None:
                misses += 1
                idx = len(row_ids) - 1
        else:
            idx = prompt_end_index(row_ids, df["prompt_len"].iloc[k] if "prompt_len" in df.columns else None)
        out.append(row_ids[: idx + 1])
    if misses:
        print(f"    warning: statement not found in {misses}/{len(ids)} rows; read at final token for those")
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.TASKS)
    ap.add_argument("--conditions", nargs="+", default=CONDITIONS)
    ap.add_argument("--readouts", nargs="+", default=["statement_end", "prompt_end", "final"])
    ap.add_argument("--layers", default="all")
    ap.add_argument("--seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--cache-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    args = ap.parse_args()

    model, tokenizer = C.load_model(C.DEEPSEEK_ID)
    layers = C.parse_layers(args.layers, model.config.num_hidden_layers + 1)

    for task in args.tasks:
        for cond in args.conditions:
            tr, te = C.load_split(cond, task, "train"), C.load_split(cond, task, "test")
            for readout in args.readouts:
                if C.already_done("exp5_readout_position", train_task=task, train_condition=cond, readout=readout):
                    print(f"  {task} {cond} {readout}: already done, skipping")
                    continue
                acts = {}
                for split, df in (("train", tr), ("test", te)):
                    ids = cut_ids(df, readout, tokenizer)
                    acts[split] = C.cached_activations(
                        args.cache_dir, f"{C.DEEPSEEK_SHORT}__{cond}__{task}__{split}__{readout}",
                        lambda ids=ids, split=split: C.extract_all_layer_activations(
                            ids, model, C.DEEPSEEK_PAD_ID, args.batch_size, f"{task} {cond} {split} {readout}"))
                rows = []
                for seed in args.seeds:
                    for layer in layers:
                        w, m = C.fit_linear_probe(acts["train"][layer], tr["y"].to_numpy(), seed)
                        auc, lo, hi = C.auroc_with_ci(te["y"], C.probe_scores(w, m, acts["test"][layer]), seed=seed)
                        rows.append(C.result_row(
                            experiment="exp5_readout_position", model=C.DEEPSEEK_SHORT,
                            train_task=task, test_task=task, train_condition=cond, test_condition=cond,
                            readout=readout, layer=layer, seed=seed, eval_target="label", subset="all",
                            n_train=len(tr), n_test=len(te), auroc=auc, ci_low=lo, ci_high=hi))
                C.append_results(rows)
                best = max(rows, key=lambda r: r["auroc"])
                print(f"  {task} {cond:<36} {readout:<14} best {best['auroc']:.3f} (layer {best['layer']})")
        summarize(task)


def summarize(task):
    df = pd.read_csv(C.RESULTS_CSV)
    df = df[(df["experiment"] == "exp5_readout_position") & (df["train_task"] == task)]
    table = (df.groupby(["train_condition", "readout", "layer"])["auroc"].mean()
               .groupby(["train_condition", "readout"]).max().unstack("readout"))
    print(f"\n{task}: best-layer in-domain AUROC by condition x readout position")
    print(table.round(3).to_string())
    table.to_csv(C.RESULTS_DIR / f"exp5_{task}_table.csv")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments/review_experiments/exp6_matched_items.py
"""
Experiment 6: compare plain and CoT on the SAME statements.

The CoT datasets lost items during filtering (generation cut-offs, verdict-extraction
failures, lexical cleaning), e.g. A2 train 700 -> 673, F5 test 593 -> 562. The plain
condition still uses every item. So "plain in-domain AUROC vs CoT in-domain AUROC" in
results_database.csv compares two different item sets, and the dropped items are not
random: they skew towards traces the model struggled with.

Every CoT statement is in the plain split of the same name (checked: no CoT train item
is in plain test or vice versa), so plain rows are matched to CoT rows by statement text.

Step 1 (CPU): write datasets/matched_items/{task}_{split}.csv with the plain statement
and label for exactly the CoT items, in CoT row order, and print what was dropped.
Step 2 (GPU): rerun in-domain (plain, CoT) and cross-condition (plain <-> CoT) probes on
the matched items only, with seeds and bootstrap CIs, next to the full-plain numbers.

Usage:
  python experiments/review_experiments/exp6_matched_items.py --prepare-only
  python experiments/review_experiments/exp6_matched_items.py --layers all \
      --cache-dir /content/drive/MyDrive/dsab_cache
"""

import argparse
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402

COT = "sentence-based-CoT"
PLAIN = "no-prompt"
MATCHED = "no-prompt-matched"
MATCHED_DIR = C.DATA_DIR / "matched_items"


def prepare(tasks):
    MATCHED_DIR.mkdir(parents=True, exist_ok=True)
    report = []
    for task in tasks:
        for split in ["train", "test"]:
            # Full plain file, never subsampled: it is the lookup table for the CoT rows.
            plain = pd.read_csv(C.CONDITION_DICT[PLAIN] / f"{task}_{split}.csv")
            plain["y"] = C.to_binary_labels(plain["label"])
            cot = C.load_split(COT, task, split)
            plain["key"] = plain["statement"].str.strip()
            cot_keys = [C.statement_text(r) for _, r in cot.iterrows()]
            first = plain.drop_duplicates("key").set_index("key")
            missing = [k for k in cot_keys if k not in first.index]
            if missing:
                raise ValueError(f"{task} {split}: {len(missing)} CoT statements not in plain, e.g. {missing[0]!r}")
            matched = first.loc[cot_keys].reset_index()
            label_clash = int((matched["y"].to_numpy() != cot["y"].to_numpy()).sum())
            matched[["key", "label"]].rename(columns={"key": "statement"}).to_csv(
                MATCHED_DIR / f"{task}_{split}.csv", index=False)

            dropped = plain[~plain["key"].isin(set(cot_keys))]
            report.append({
                "task": task, "split": split, "plain": len(plain), "cot": len(cot),
                "dropped": len(dropped),
                "dropped_true_share": round(float(dropped["y"].mean()), 3) if len(dropped) else float("nan"),
                "kept_true_share": round(float(cot["y"].mean()), 3),
                "label_mismatches": label_clash,
                "duplicate_statements_in_plain": int(plain["key"].duplicated().sum()),
            })
    report = pd.DataFrame(report)
    print(report.to_string(index=False))
    C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    report.to_csv(C.RESULTS_DIR / "exp6_dropped_items.csv", index=False)


def load_matched(task, split):
    df = pd.read_csv(MATCHED_DIR / f"{task}_{split}.csv")
    df["y"] = C.to_binary_labels(df["label"])
    return df


def run(tasks, layers_arg, seeds, cache_dir, batch_size):
    model, tokenizer = C.load_model(C.DEEPSEEK_ID)
    layers = C.parse_layers(layers_arg, model.config.num_hidden_layers + 1)

    def acts(cond, task, split, df):
        return C.cached_activations(
            cache_dir, f"{C.DEEPSEEK_SHORT}__{cond}__{task}__{split}__final",
            lambda: C.extract_all_layer_activations(C.ids_for(df, tokenizer), model, C.DEEPSEEK_PAD_ID,
                                                    batch_size, f"{task} {cond} {split}"))

    for task in tasks:
        if C.already_done("exp6_matched_items", train_task=task):
            print(f"\n{task}: already done, skipping")
            continue
        data = {
            PLAIN: (C.load_split(PLAIN, task, "train"), C.load_split(PLAIN, task, "test")),
            MATCHED: (load_matched(task, "train"), load_matched(task, "test")),
            COT: (C.load_split(COT, task, "train"), C.load_split(COT, task, "test")),
        }
        A = {c: (acts(c, task, "train", tr), acts(c, task, "test", te)) for c, (tr, te) in data.items()}
        pairs = [(PLAIN, PLAIN), (MATCHED, MATCHED), (COT, COT), (MATCHED, COT), (COT, MATCHED)]
        rows = []
        for seed in seeds:
            for layer in layers:
                for tr_c, te_c in pairs:
                    y_tr = data[tr_c][0]["y"].to_numpy()
                    y_te = data[te_c][1]["y"].to_numpy()
                    w, m = C.fit_linear_probe(A[tr_c][0][layer], y_tr, seed)
                    auc, lo, hi = C.auroc_with_ci(y_te, C.probe_scores(w, m, A[te_c][1][layer]), seed=seed)
                    rows.append(C.result_row(
                        experiment="exp6_matched_items", model=C.DEEPSEEK_SHORT, train_task=task, test_task=task,
                        train_condition=tr_c, test_condition=te_c, readout="final", layer=layer, seed=seed,
                        eval_target="label", subset="all", n_train=len(y_tr), n_test=len(y_te),
                        auroc=auc, ci_low=lo, ci_high=hi))
        C.append_results(rows)
        df = pd.DataFrame(rows)
        best = df.groupby(["train_condition", "test_condition", "layer"])["auroc"].mean() \
                 .groupby(["train_condition", "test_condition"]).max()
        print(f"\n{task}: best-layer AUROC")
        print(best.round(3).to_string())


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.TASKS)
    ap.add_argument("--prepare-only", action="store_true")
    ap.add_argument("--layers", default="all")
    ap.add_argument("--seeds", nargs="+", type=int, default=[0, 1, 2])
    ap.add_argument("--cache-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    args = ap.parse_args()

    prepare(args.tasks)
    if not args.prepare_only:
        run(args.tasks, args.layers, args.seeds, args.cache_dir, args.batch_size)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments/review_experiments/exp7_granite_conditions.py
"""
Experiment 7: give the Granite run something to compare against.

results_database.csv has Granite (granite-4.2-8b) rows for sentence-based-CoT only, so
Granite cannot support a "CoT changes the truth direction" claim on its own. This adds
the plain conditions and the cross-condition transfer on Granite.

How the existing Granite run built its inputs (Extract_Layers_Granite_Ablation.ipynb):
it DECODES DeepSeek's token ids back to text and re-tokenizes with Granite's tokenizer.
So the "Granite CoT" condition is DeepSeek's reasoning trace read by Granite, including
DeepSeek's chat markers ('<｜User｜>', '<｜Assistant｜>', '<think>') as plain text. That
is a fair test of "does externalized reasoning help a different model", and the paper
should describe it that way. This script keeps that input exactly (so its CoT numbers
reproduce the existing rows) and adds:

  no-prompt                       the bare statement, Granite tokenizer
  no-prompt-granite-template      the statement inside Granite's OWN chat template
                                  (apply_chat_template, add_generation_prompt=True)
  sentence-based-CoT              unchanged DeepSeek-trace input (as the existing run)
  sentence-based-CoT-clean        same trace with the DeepSeek marker strings removed,
                                  to check the markers are not driving the result

Cross-condition: no-prompt <-> sentence-based-CoT in both directions, as in Figure 3.

Usage (A100/L4 class GPU; Granite 8B in fp16 needs ~17 GB):
  python experiments/review_experiments/exp7_granite_conditions.py --layers 0-40:2 \
      --cache-dir /content/drive/MyDrive/dsab_cache
"""

import argparse
import sys
from ast import literal_eval
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402

GRANITE_ID = "ibm-granite/granite-4.2-8b"
GRANITE_SHORT = "granite-4.2-8b"
DEEPSEEK_MARKER_STRINGS = ["<｜begin▁of▁sentence｜>", "<｜User｜>", "<｜Assistant｜>"]


def granite_inputs(condition, df, g_tok, ds_tok):
    """Granite token ids for each row under `condition`."""
    if condition == "no-prompt":
        return g_tok(df["statement"].tolist())["input_ids"]
    if condition == "no-prompt-granite-template":
        # Render to text, then tokenize: apply_chat_template(tokenize=True) returns a dict
        # (input_ids, attention_mask) in recent transformers, not a list of ids.
        texts = [
            g_tok.apply_chat_template([{"role": "user", "content": s}], add_generation_prompt=True, tokenize=False)
            for s in df["statement"]
        ]
        return g_tok(texts, add_special_tokens=False)["input_ids"]
    # CoT: decode DeepSeek ids to text, as the existing Granite notebook does.
    texts = [ds_tok.decode(literal_eval(x) if isinstance(x, str) else x) for x in df["extracted_statement_ids"]]
    if condition == "sentence-based-CoT-clean":
        for marker in DEEPSEEK_MARKER_STRINGS:
            texts = [t.replace(marker, "") for t in texts]
        texts = [t.replace("<think>\n", "") for t in texts]
    return g_tok(texts)["input_ids"]


def source_condition(condition):
    return {"no-prompt": "no-prompt", "no-prompt-granite-template": "no-prompt",
            "sentence-based-CoT": "sentence-based-CoT", "sentence-based-CoT-clean": "sentence-based-CoT"}[condition]


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.TASKS)
    ap.add_argument("--conditions", nargs="+",
                    default=["no-prompt", "no-prompt-granite-template", "sentence-based-CoT", "sentence-based-CoT-clean"])
    ap.add_argument("--layers", default="all")
    ap.add_argument("--seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--cache-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    args = ap.parse_args()

    from transformers import AutoTokenizer
    ds_tok = AutoTokenizer.from_pretrained(C.DEEPSEEK_ID)
    model, g_tok = C.load_model(GRANITE_ID)
    pad_id = g_tok.pad_token_id
    layers = C.parse_layers(args.layers, model.config.num_hidden_layers + 1)

    for task in args.tasks:
        if C.already_done("exp7_granite_conditions", train_task=task):
            print(f"\n{task}: already done, skipping")
            continue
        data, acts = {}, {}
        for cond in args.conditions:
            tr = C.load_split(source_condition(cond), task, "train")
            te = C.load_split(source_condition(cond), task, "test")
            data[cond] = (tr, te)
            acts[cond] = tuple(
                C.cached_activations(
                    args.cache_dir, f"{GRANITE_SHORT}__{cond}__{task}__{split}__final",
                    lambda df=df, split=split, cond=cond: C.extract_all_layer_activations(
                        granite_inputs(cond, df, g_tok, ds_tok), model, pad_id, args.batch_size,
                        f"{task} {cond} {split}"))
                for split, df in (("train", tr), ("test", te)))

        pairs = [(c, c) for c in args.conditions]
        if {"no-prompt", "sentence-based-CoT"} <= set(args.conditions):
            pairs += [("no-prompt", "sentence-based-CoT"), ("sentence-based-CoT", "no-prompt")]
        rows = []
        for seed in args.seeds:
            for layer in layers:
                for tr_c, te_c in pairs:
                    y_tr = data[tr_c][0]["y"].to_numpy()
                    y_te = data[te_c][1]["y"].to_numpy()
                    w, m = C.fit_linear_probe(acts[tr_c][0][layer], y_tr, seed)
                    auc, lo, hi = C.auroc_with_ci(y_te, C.probe_scores(w, m, acts[te_c][1][layer]), seed=seed)
                    rows.append(C.result_row(
                        experiment="exp7_granite_conditions", model=GRANITE_SHORT, train_task=task, test_task=task,
                        train_condition=tr_c, test_condition=te_c, readout="final", layer=layer, seed=seed,
                        eval_target="label", subset="all", n_train=len(y_tr), n_test=len(y_te),
                        auroc=auc, ci_low=lo, ci_high=hi))
        C.append_results(rows)
        best = pd.DataFrame(rows).groupby(["train_condition", "test_condition", "layer"])["auroc"].mean() \
                 .groupby(["train_condition", "test_condition"]).max()
        print(f"\n{task} (Granite): best-layer AUROC")
        print(best.round(3).to_string())
        del acts


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments/review_experiments/exp9_lexical_baselines.py
"""
Experiment 9: put the word-only baselines in the same table as the probes.

lexical_shortcut_baseline.py writes its own CSVs, separate from results_database.csv,
and runs on the CoT data only. This reruns the same baselines (same vectorizer settings,
same LogisticRegression) on three kinds of input, adds bootstrap CIs, writes them to
results/review_results.csv in the same row format as the probes, and builds the main
comparison table the paper needs.

Inputs:
  cot-full        the full extracted CoT prefix (prompt + statement + reasoning). This is
                  what lexical_shortcut_baseline.py uses, so it should reproduce those numbers.
  cot-reasoning   only the reasoning text after the prompt. The prompt is the same for every
                  item except the statement, so this isolates what the REASONING words give away.
  plain-statement the bare statement. Can the statement's own words (city and country
                  names, numbers) predict its truth without any model at all?
Plus the final-token one-hot baseline on the CoT prefix, as in the original script.

Output:
  results/main_table.csv and .md: per task, best-layer probe AUROC (in-domain plain, in-domain
  CoT, plain->CoT, CoT->plain, from results_database.csv) next to the four baselines.

Usage (CPU, ~1 minute):
  python experiments/review_experiments/exp9_lexical_baselines.py
"""

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

sys.path.insert(0, str(Path(__file__).resolve().parent))
import common as C  # noqa: E402

COT = "sentence-based-CoT"
PLAIN = "no-prompt"
TOKEN_PATTERN = r"(?u)\b[A-Za-z0-9']+\b"  # same as lexical_shortcut_baseline.py


def reasoning_only(text):
    return text.split("<think>", 1)[-1]


def last_word(text):
    tokens = re.findall(r"[A-Za-z']+|\d+", str(text).rstrip())
    return tokens[-1].lower() if tokens else "<EMPTY>"


def tfidf_scores(train_texts, y_train, test_texts, seed):
    vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 1), token_pattern=TOKEN_PATTERN)
    clf = LogisticRegression(max_iter=1000, random_state=seed)
    clf.fit(vec.fit_transform(train_texts), y_train)
    return clf.predict_proba(vec.transform(test_texts))[:, 1]


def final_token_scores(train_texts, y_train, test_texts, seed):
    enc = OneHotEncoder(handle_unknown="ignore")
    Xtr = enc.fit_transform(np.array([last_word(t) for t in train_texts]).reshape(-1, 1))
    Xte = enc.transform(np.array([last_word(t) for t in test_texts]).reshape(-1, 1))
    clf = LogisticRegression(max_iter=1000, random_state=seed).fit(Xtr, y_train)
    return clf.predict_proba(Xte)[:, 1]


def run_baselines(tasks, seed):
    rows = []
    for task in tasks:
        ctr, cte = C.load_split(COT, task, "train"), C.load_split(COT, task, "test")
        ptr, pte = C.load_split(PLAIN, task, "train"), C.load_split(PLAIN, task, "test")
        cot_full = (ctr["extracted_statement_texts"].fillna(""), cte["extracted_statement_texts"].fillna(""))
        cot_reason = (cot_full[0].map(reasoning_only), cot_full[1].map(reasoning_only))
        variants = [
            ("tfidf-bow", "cot-full", cot_full, ctr, cte, tfidf_scores),
            ("tfidf-bow", "cot-reasoning", cot_reason, ctr, cte, tfidf_scores),
            ("tfidf-bow", "plain-statement", (ptr["statement"], pte["statement"]), ptr, pte, tfidf_scores),
            ("final-token", "cot-full", cot_full, ctr, cte, final_token_scores),
        ]
        for model_name, readout, (tr_text, te_text), tr, te, fn in variants:
            scores = fn(tr_text.tolist(), tr["y"], te_text.tolist(), seed)
            auc, lo, hi = C.auroc_with_ci(te["y"], scores, seed=seed)
            cond = PLAIN if readout == "plain-statement" else COT
            rows.append(C.result_row(
                experiment="exp9_lexical_baselines", model=model_name, train_task=task, test_task=task,
                train_condition=cond, test_condition=cond, readout=readout, layer=-1, seed=seed,
                eval_target="label", subset="all", n_train=len(tr), n_test=len(te),
                auroc=auc, ci_low=lo, ci_high=hi))
    C.append_results(rows)
    return pd.DataFrame(rows)


def probe_best(db, train_c, test_c):
    sub = db[(db["model"] == C.DEEPSEEK_SHORT) & (db["train_task"] == db["test_task"])
             & (db["train_condition"] == train_c) & (db["test_condition"] == test_c)]
    idx = sub.groupby("train_task")["auroc"].idxmax()
    best = sub.loc[idx].set_index("train_task")
    return best.apply(lambda r: f"{r.auroc:.3f} (L{int(r.layer)})", axis=1)


def build_table(baselines):
    db = pd.read_csv(C.REPO_ROOT / "experiments" / "results_database.csv")
    table = pd.DataFrame(index=C.TASKS)
    table["probe plain"] = probe_best(db, PLAIN, PLAIN)
    table["probe CoT"] = probe_best(db, COT, COT)
    table["probe plain->CoT"] = probe_best(db, PLAIN, COT)
    table["probe CoT->plain"] = probe_best(db, COT, PLAIN)

    def fmt(r):
        return f"{r.auroc:.3f} [{r.ci_low:.2f}, {r.ci_high:.2f}]"

    for (model_name, readout), label in [
        (("tfidf-bow", "plain-statement"), "TF-IDF statement"),
        (("tfidf-bow", "cot-reasoning"), "TF-IDF CoT reasoning"),
        (("tfidf-bow", "cot-full"), "TF-IDF CoT full"),
        (("final-token", "cot-full"), "final word CoT"),
    ]:
        sub = baselines[(baselines["model"] == model_name) & (baselines["readout"] == readout)]
        table[label] = sub.set_index("train_task").apply(fmt, axis=1)

    table.index.name = "task"
    table.to_csv(C.RESULTS_DIR / "main_table.csv")
    try:
        md = table.to_markdown()
    except ImportError:
        md = table.to_string()
    (C.RESULTS_DIR / "main_table.md").write_text(
        "Probe columns: best-layer AUROC from results_database.csv (DeepSeek-R1-Distill-8B), "
        "layer in brackets. Probe values have no CI yet: rerun with exp6 seeds/bootstrap.\n"
        "Baselines: AUROC with 95% bootstrap CI over test items.\n\n" + md + "\n")
    print(md)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=C.TASKS)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()
    baselines = run_baselines(args.tasks, args.seed)
    build_table(baselines)


if __name__ == "__main__":
    main()

## 5. Sanity check (~5 min)

Reruns two numbers that already exist in `results_database.csv` (A2, layers 22 and 30) with
the new code, written to a scratch folder so it does not count as a real result. The new
numbers should match to about two decimal places (fp16 adds small noise). If they are far
off, stop and check before spending hours on the rest.

In [ ]:
import pandas as pd
SANITY = '/content/sanity_results'
!rm -rf {SANITY}
!DSAB_RESULTS_DIR={SANITY} python experiments/review_experiments/exp6_matched_items.py --tasks A2 --layers 22,30 --seeds 0

new = pd.read_csv(f'{SANITY}/review_results.csv')
old = pd.read_csv('experiments/results_database.csv')
old = old[(old.model == 'deepseek-r1-distill-8b') & (old.train_task == 'A2') & (old.test_task == 'A2')]
rows = []
for cond in ['no-prompt', 'sentence-based-CoT']:
    for layer in [22, 30]:
        n = new[(new.train_condition == cond) & (new.test_condition == cond) & (new.layer == layer)].auroc
        o = old[(old.train_condition == cond) & (old.test_condition == cond) & (old.layer == layer)].auroc
        rows.append({'condition': cond, 'layer': layer, 'existing': round(o.iloc[0], 3), 'new run': round(n.iloc[0], 3)})
check = pd.DataFrame(rows)
check['diff'] = (check['new run'] - check['existing']).abs()
print(check.to_string(index=False))
print('\nOK, matches the existing results' if check['diff'].max() < 0.03 else
      '\nWARNING: differs from the existing results by more than 0.03. Check before going on.')

## 6. Experiment 3: does the probe follow the truth or the model's answer? (~45 min)

Factual tasks only (arithmetic has almost no model errors). Look at the printed lines for
`model_wrong`: AUROC against the TRUE label on items the model answered wrongly.
**Above 0.5** with an interval that excludes 0.5 = the probe follows the truth.
**Below 0.5** = it follows the model's answer.

In [ ]:
!python experiments/review_experiments/exp3_probe_vs_verdict.py --tasks F0 F1 F2 F3 F4 F5 --cache-dir {CACHE}/exp3
!rm -rf {CACHE}/exp3

## 7. Experiment 6: plain vs CoT on exactly the same statements (~1 h)

Compares `no-prompt` (all items) with `no-prompt-matched` (only the items that survived CoT
filtering). If they differ, the filtering, which dropped mostly TRUE arithmetic statements,
is affecting the plain vs CoT comparison.

In [ ]:
!python experiments/review_experiments/exp6_matched_items.py --cache-dir {CACHE}/exp6
!rm -rf {CACHE}/exp6

## 8a. Experiment 4: reading partway through the reasoning (~1.5 h)

The key question is the arithmetic curve: if the probe is already high at 25-75% of the
trace, where the words give nothing away (TF-IDF ~0.5 there), the model holds the answer
before it writes it. No cache here: 5 fractions x 9 tasks would need ~17 GB.

In [ ]:
!python experiments/review_experiments/exp4_early_readout.py --layers 0-32:2

## 8b. Experiment 5: same readout position for every condition (~1 h)

Separates "the instructions changed things" from "the readout moved off the statement".
The `statement_end` column should be nearly the same across conditions; that is a check.
Compare conditions within the `prompt_end` column.

In [ ]:
!python experiments/review_experiments/exp5_readout_position.py --tasks A1 A2 A3 F3 F4 F5 --layers 0-32:2

## 8c. Experiment 7: Granite (~1.5 h, ~17 GB)

Adds the plain conditions and plain <-> CoT transfer on Granite. Each experiment runs in
its own process, so DeepSeek's memory is already free by now.

In [ ]:
!python experiments/review_experiments/exp7_granite_conditions.py --layers 0-40:2 --cache-dir {CACHE}/exp7
!rm -rf {CACHE}/exp7

## 9. Main table, summaries, and a zip of everything

In [ ]:
!python experiments/review_experiments/exp9_lexical_baselines.py > /dev/null
import sys, glob
from IPython.display import Image, Markdown, display
sys.path.insert(0, 'experiments/review_experiments')
import common as C
res = pd.read_csv(C.RESULTS_CSV)
print(res.groupby('experiment').size().to_string(), '\n')

def best(df, by):
    g = df.groupby(by + ['layer'])[['auroc', 'ci_low', 'ci_high']].mean().reset_index()
    return g.loc[g.groupby(by)['auroc'].idxmax()].set_index(by).round(3)

display(Markdown('### Main table'))
display(Markdown(open(f'{RESULTS}/main_table.md').read()))

e3 = res[(res.experiment == 'exp3_probe_vs_verdict') & (res.eval_target == 'label')]
if len(e3):
    display(Markdown('### Exp 3: AUROC vs TRUE label on items the model got wrong (>0.5 = follows truth). Layer chosen on all items.'))
    g = e3.groupby(['train_task', 'train_condition', 'test_condition', 'subset', 'layer'])[['auroc', 'ci_low', 'ci_high', 'n_test']].mean().reset_index()
    allm = g[g.subset == 'all']
    pick = allm.loc[allm.groupby(['train_task', 'train_condition', 'test_condition'])['auroc'].idxmax(),
                    ['train_task', 'train_condition', 'test_condition', 'layer', 'auroc']].rename(columns={'auroc': 'auroc_all_items'})
    wrong = g[g.subset == 'model_wrong'].drop(columns='subset').rename(columns={'auroc': 'auroc_model_wrong'})
    display(pick.merge(wrong, on=['train_task', 'train_condition', 'test_condition', 'layer']).round(3))

for exp, title in [('exp6_matched_items', 'Exp 6: matched items, best layer'),
                   ('exp7_granite_conditions', 'Exp 7: Granite, best layer')]:
    sub = res[res.experiment == exp]
    if len(sub):
        display(Markdown(f'### {title}'))
        display(best(sub, ['train_task', 'train_condition', 'test_condition']))

for f in sorted(glob.glob(f'{RESULTS}/exp5_*_table.csv')):
    display(Markdown(f'### {f.split("/")[-1]}'))
    display(pd.read_csv(f, index_col=0).round(3))

if os.path.exists(f'{RESULTS}/exp4_early_readout.png'):
    display(Markdown('### Exp 4'))
    display(Image(f'{RESULTS}/exp4_early_readout.png'))

!cd /content/drive/MyDrive && zip -qr /content/dsab_review_results.zip dsab_review_results
from google.colab import files
files.download('/content/dsab_review_results.zip')